# 04 — SCD Type 4: Separate History Table

Schema:

```text
current table -> latest state
history table -> old values for changed rows
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_4_history_table")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
])

history_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("valid_from", DateType(), True),
    StructField("valid_to", DateType(), True),
])

incoming_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("change_date_raw", StringType(), False),
])

current = spark.createDataFrame(
    [("u1", "SK", "retail"), ("u2", "CZ", "retail")],
    current_schema,
)

history = spark.createDataFrame([], history_schema)

incoming = (
    spark.createDataFrame(
        [
            ("u1", "SK", "premium", "2026-02-01"),
            ("u2", "CZ", "retail", "2026-02-01"),
            ("u3", "DE", "retail", "2026-02-01"),
        ],
        incoming_schema,
    )
    .withColumn("change_date", F.to_date("change_date_raw"))
    .drop("change_date_raw")
)

current.show(truncate=False)
history.show(truncate=False)
incoming.show(truncate=False)

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |SK     |retail |
|u2     |CZ     |retail |
+-------+-------+-------+

+-------+-------+-------+----------+--------+
|user_id|country|segment|valid_from|valid_to|
+-------+-------+-------+----------+--------+
+-------+-------+-------+----------+--------+

+-------+-------+-------+-----------+
|user_id|country|segment|change_date|
+-------+-------+-------+-----------+
|u1     |SK     |premium|2026-02-01 |
|u2     |CZ     |retail |2026-02-01 |
|u3     |DE     |retail |2026-02-01 |
+-------+-------+-------+-----------+



In [3]:
joined = (
    incoming.alias("i")
    .join(current.alias("c"), on="user_id", how="left")
    .select(
        "user_id",
        F.col("i.country").alias("new_country"),
        F.col("i.segment").alias("new_segment"),
        F.col("i.change_date"),
        F.col("c.country").alias("old_country"),
        F.col("c.segment").alias("old_segment"),
    )
    .withColumn("is_new_key", F.col("old_country").isNull())
    .withColumn(
        "is_changed",
        (~F.col("is_new_key")) &
        ((F.col("new_country") != F.col("old_country")) | (F.col("new_segment") != F.col("old_segment")))
    )
)

new_history_rows = (
    joined
    .filter("is_changed")
    .select(
        "user_id",
        F.col("old_country").alias("country"),
        F.col("old_segment").alias("segment"),
        F.lit(None).cast(DateType()).alias("valid_from"),
        F.date_sub(F.col("change_date"), 1).alias("valid_to"),
    )
)

history_updated = history.unionByName(new_history_rows)
history_updated.show(truncate=False)

+-------+-------+-------+----------+----------+
|user_id|country|segment|valid_from|valid_to  |
+-------+-------+-------+----------+----------+
|u1     |SK     |retail |NULL      |2026-01-31|
+-------+-------+-------+----------+----------+



In [4]:
updated_existing = (
    current.alias("c")
    .join(incoming.alias("i"), on="user_id", how="left")
    .select(
        "user_id",
        F.coalesce(F.col("i.country"), F.col("c.country")).alias("country"),
        F.coalesce(F.col("i.segment"), F.col("c.segment")).alias("segment"),
    )
)

new_current_rows = (
    incoming.join(current.select("user_id"), on="user_id", how="left_anti")
    .select("user_id", "country", "segment")
)

current_updated = updated_existing.unionByName(new_current_rows).orderBy("user_id")
current_updated.show(truncate=False)

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |SK     |premium|
|u2     |CZ     |retail |
|u3     |DE     |retail |
+-------+-------+-------+



In [5]:
totals = spark.createDataFrame(
    [
        ("current_before", current.count()),
        ("incoming_rows", incoming.count()),
        ("history_rows_added", new_history_rows.count()),
        ("current_after", current_updated.count()),
        ("history_after", history_updated.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)
totals.show(truncate=False)

+------------------+-----+
|metric            |value|
+------------------+-----+
|current_before    |2    |
|incoming_rows     |3    |
|history_rows_added|1    |
|current_after     |3    |
|history_after     |1    |
+------------------+-----+



In [6]:
spark.stop()